# EcoSignal — Repair Satellite Preview
This notebook rebuilds the preview from **existing satellite outputs**. It makes **no Copernicus API calls**, needs **no OAuth credentials**, and requests **no external basemap tiles**. A plain background is intentional; the map retains your change overlays, boundary, pan/zoom and layer controls. Leaflet assets still require internet.

The reported TIFF warnings concern colour-channel metadata. They did not stop your completed run. This repair checks band counts and grid alignment before rendering; it does not hide raster warnings or rewrite source imagery.

In [ ]:
%pip -q install rasterio numpy scipy pyproj shapely pandas folium matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # Select your satellite output ZIP
zip_name = next(iter(uploaded))
extract_dir = Path("/content/satellite_results")

with zipfile.ZipFile(zip_name) as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)

## Select the completed output folder
The default matches your reported run. Change it only if your results are elsewhere.

In [ ]:
from pathlib import Path
import json
OUTPUT = Path('/content/satellite_results')
if not (OUTPUT/'manifest.json').is_file():
    raise FileNotFoundError(f'No manifest.json in {OUTPUT}. Set OUTPUT to your completed run folder.')
manifest = json.loads((OUTPUT/'manifest.json').read_text())
if not manifest.get('completed'):
    raise ValueError('This run is incomplete. Complete processing before repairing the preview.')

## Load the preview functions
No imagery is downloaded or processed in this cell.

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import shape, mapping
NAMES = ['NDVI','NDMI','BSI','MNDWI']
def load_aoi(path):
    data = json.loads(Path(path).read_text())
    if data['type'] == 'FeatureCollection':
        from shapely.ops import unary_union
        return unary_union([shape(f['geometry']) for f in data['features']])
    if data['type'] == 'Feature':
        return shape(data['geometry'])
    return shape(data)

def preview(root, manifest):
    import folium
    from rasterio.warp import reproject, calculate_default_transform, Resampling
    import matplotlib
    root=Path(root);g=load_aoi(root/'study_area.geojson')
    m=folium.Map(location=[g.centroid.y,g.centroid.x],zoom_start=10,tiles=None)
    latest=max(o['week'] for o in manifest['outputs'] if o['kind']=='change')
    groups=[folium.FeatureGroup(name='Weekly '+n+' change',show=(j==0)) for j,n in enumerate(NAMES)]
    for entry in manifest['outputs']:
        if entry['kind']!='change' or entry['week']!=latest: continue
        with rasterio.open(root/entry['file']) as ds:
            affine,w,h=calculate_default_transform(ds.crs,3857,ds.width,ds.height,*ds.bounds)
            for j,group in enumerate(groups):
                dest=np.full((h,w),np.nan,dtype=np.float32)
                reproject(source=rasterio.band(ds,j+1),destination=dest,src_transform=ds.transform,
                    src_crs=ds.crs,dst_transform=affine,dst_crs=3857,src_nodata=np.nan,dst_nodata=np.nan,resampling=Resampling.nearest)
                rgba=(matplotlib.colormaps['RdYlGn'](np.clip((np.nan_to_num(dest)+.3)/.6,0,1))*255).astype(np.uint8)
                rgba[:,:,3]=np.where(np.isfinite(dest),170,0).astype(np.uint8)
                west,south,east,north=transform_bounds(3857,4326,*rasterio.transform.array_bounds(h,w,affine))
                folium.raster_layers.ImageOverlay(rgba,bounds=[[south,west],[north,east]],name=entry['tile']).add_to(group)
    for group in groups: group.add_to(m)
    folium.GeoJson(mapping(g),name='Study boundary').add_to(m)
    folium.LayerControl().add_to(m)
    m.fit_bounds([[g.bounds[1],g.bounds[0]],[g.bounds[3],g.bounds[2]]])
    label=f'Satellite overlays; no external basemap. Week {latest} minus previous week. Red: decrease; yellow: near zero; green: increase. Display scale −0.3 to +0.3; transparent: no comparison. BSI increase means stronger exposed-ground signal, not improvement.'
    m.get_root().html.add_child(folium.Element('<div style="position:fixed;bottom:10px;left:10px;z-index:9999;background:white;padding:8px;max-width:450px">'+label+'</div>'))
    m.save(str(root/'preview.html'));return m


## Check files and rebuild the preview

In [ ]:
weekly = [e for e in manifest['outputs'] if e['kind']=='weekly']
changes = [e for e in manifest['outputs'] if e['kind']=='change']
if not changes:
    raise ValueError('No change rasters found. At least two weeks are needed.')
reference = {}
for e in weekly + changes:
    with rasterio.open(OUTPUT/e['file']) as ds:
        expected = 8 if e['kind']=='weekly' else 4
        if ds.count != expected:
            raise ValueError(f"Unexpected band count in {e['file']}: {ds.count}")
        grid = (ds.crs.to_string(), tuple(ds.transform), ds.width, ds.height)
        if e['tile'] in reference and reference[e['tile']] != grid:
            raise ValueError(f"Grid mismatch for {e['tile']}")
        reference[e['tile']] = grid
print('PASS: expected bands and matching grids.')
from IPython.display import display
m = preview(OUTPUT, manifest)
html = (OUTPUT/'preview.html').read_text()
assert 'tile.openstreetmap.org' not in html
assert 'L.tileLayer(' not in html
print('PASS: preview contains no external basemap tile requests.')
display(m)
print('Saved:', OUTPUT/'preview.html')

## Download the repaired preview and integration ZIP
The ZIP excludes cached source imagery and includes the repaired preview.

In [ ]:
from google.colab import files
import zipfile
bundle = Path('/content') / (OUTPUT.name + '_preview_fixed.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and 'cache' not in p.relative_to(OUTPUT).parts:
            z.write(p, p.relative_to(OUTPUT))
files.download(str(bundle))